# Positioning table (3-seed, all methods) + generality tests (Theil, log-barrier)

Closes review items #1, #2, #3, #5:

- **#3**: replaces single-seed Gated/L1 rows in the positioning table with the same 3-seed protocol Top-k and Rate-KL already get.
- **#5**: adds `mean_active_magnitude` for Rate-KL (and every method), closing the missing-control gap.
- **#1**: Theil index through the same purity/importance-concentration diagnostic used on Gini -- second instance of the general hub-pathology claim.
- **#2**: log-barrier + budget penalty (identity-tracked, boundary-escalating, but NOT KL-shaped) -- second instantiation of the design principle behind Rate-KL, testing whether the principle is separable from the specific formula.

A gradient-magnitude sanity check (Theil at a hub point vs.\ a genuinely varied selective point) already confirmed the vanishing-gradient prediction analytically before this notebook runs any training -- see `experiments/theil_diagnosis.py`'s docstring.

**Before running:** Runtime -> Change runtime type -> GPU.

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go to Runtime > Change runtime type > GPU)')

CUDA available: True
Device: NVIDIA L4


In [2]:
!git clone https://github.com/willkn/SAE-Gini.git
%cd SAE-Gini/experiments

Cloning into 'SAE-Gini'...
remote: Enumerating objects: 253, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 253 (delta 64), reused 75 (delta 25), pack-reused 125 (from 2)
Receiving objects: 100% (253/253), 246.01 MiB | 2.82 MiB/s, done.
Resolving deltas: 100% (77/77), done.
/content/SAE-Gini/experiments


## Part 1: Full positioning replication (3 seeds, Fashion-MNIST)
Trains Tuned L1, Gated, Gini, Top-k, Rate-KL (lambda=0.001, 0.01) per seed -- 5 models x 3 seeds = 15 trainings.

In [3]:
!python run_full_positioning_replication.py --dataset fashion_mnist --seeds 0 1 2 --rho 0.09 --lambdas 0.001 0.01

100% 26.4M/26.4M [00:02<00:00, 9.79MB/s]
100% 29.5k/29.5k [00:00<00:00, 147kB/s]
100% 4.42M/4.42M [00:01<00:00, 2.74MB/s]
100% 5.15k/5.15k [00:00<00:00, 28.7MB/s]
[fashion_mnist seed=0] Training Tuned L1...
[fashion_mnist seed=0] Training Gated SAE...
[fashion_mnist seed=0] Training Gini SAE...
[fashion_mnist seed=0] Training Top-K (k=92)...
[fashion_mnist seed=0] Training RateKL_lam0.001...
[fashion_mnist seed=0] Training RateKL_lam0.01...

=== fashion_mnist seed=0: full positioning ===
Model             Sparsity      MSE  Magnitude  MeanPurity  Top10Purity   Ablate    Clamp
TunedL1              0.546   0.0019      0.242       0.050        0.000   0.0233   0.0466
GatedSAE             0.767   0.0049      0.166       0.109        0.007   0.0264   0.0529
Gini                 0.912   0.0088      2.102       0.148        0.001   0.1318   0.2636
TopK                 0.910   0.0060      0.743       0.272        0.030   0.0636   0.1272
RateKL_lam0.001      0.903   0.0096      0.343       0.60

In [4]:
import json
with open('results/full_positioning/fashion_mnist_summary.json') as f:
    summary = json.load(f)
print(f"{'Model':16s} {'Sparsity':>11s} {'MSE':>13s} {'Magnitude':>14s} {'MeanPurity':>14s} {'Top10Purity':>14s} {'Ablate':>14s} {'Clamp':>14s}")
for model, m in summary.items():
    def fmt(key, prec=4):
        v = m.get(key)
        return f"{v['mean']:.{prec}f}+/-{v['std']:.{prec}f}" if v else 'n/a'
    print(f"{model:16s} {fmt('relative_sparsity', 3):>11s} {fmt('mse'):>13s} {fmt('mean_active_magnitude', 3):>14s} "
          f"{fmt('mean_purity', 3):>14s} {fmt('mean_purity_top10pct_by_importance', 3):>14s} "
          f"{fmt('steering_impact_ablate'):>14s} {fmt('steering_impact_clamp'):>14s}")

Model               Sparsity           MSE      Magnitude     MeanPurity    Top10Purity         Ablate          Clamp
GatedSAE         0.760+/-0.007 0.0043+/-0.0005  0.166+/-0.001  0.101+/-0.007  0.006+/-0.000 0.0257+/-0.0006 0.0515+/-0.0013
Gini             0.913+/-0.002 0.0088+/-0.0001  2.120+/-0.024  0.146+/-0.017  0.001+/-0.000 0.1261+/-0.0109 0.2523+/-0.0218
RateKL_lam0.001  0.905+/-0.003 0.0096+/-0.0001  0.335+/-0.007  0.605+/-0.005  0.442+/-0.021 0.0781+/-0.0027 0.1561+/-0.0054
RateKL_lam0.01   0.910+/-0.007 0.0163+/-0.0005  0.338+/-0.015  0.683+/-0.009  0.534+/-0.015 0.1496+/-0.0030 0.2991+/-0.0060
TopK             0.910+/-0.000 0.0059+/-0.0000  0.739+/-0.005  0.275+/-0.006  0.034+/-0.005 0.0665+/-0.0033 0.1329+/-0.0066
TunedL1          0.547+/-0.008 0.0020+/-0.0002  0.240+/-0.006  0.047+/-0.006  0.000+/-0.000 0.0229+/-0.0010 0.0458+/-0.0020


## Part 2: Theil index generality test (Fashion-MNIST)
Second instance of the hub-pathology claim: does a differently-shaped inequality measure (Theil, not Gini) produce the same hub signature?

In [ ]:
!python theil_diagnosis.py --dataset fashion_mnist --seed 0 --lambdas 0.0005 0.001 0.002 0.004 0.008 --matched-k 92

## Part 3: Log-barrier second instantiation (Fashion-MNIST)
Tests whether the identity-tracked + boundary-escalating DESIGN PRINCIPLE (not the specific KL formula) is what avoids the hub pathology, using a functionally unrelated penalty (interior-point log-barrier) that shares only those two properties.

In [6]:
!python rate_barrier_sae.py --dataset fashion_mnist --seed 0 --rho 0.09 --lambda-pairs 0.001,0.1 0.005,0.5 0.01,1.0

[fashion_mnist seed=0] Training RateBarrier_b0.001_u0.1...
[fashion_mnist seed=0] Training RateBarrier_b0.005_u0.5...
[fashion_mnist seed=0] Training RateBarrier_b0.01_u1.0...
[fashion_mnist seed=0] Training TopK reference (k=92)...

=== fashion_mnist seed=0: log-barrier second instantiation ===
Model                     Sparsity      MSE  Top10Purity   Ablate    Clamp
RateBarrier_b0.001_u0.1      0.784   0.0076        0.333   0.0510   0.1020
RateBarrier_b0.005_u0.5      0.770   0.0111        0.406   0.0930   0.1860
RateBarrier_b0.01_u1.0       0.781   0.0114        0.385   0.1020   0.2041
TopK_reference               0.910   0.0060        0.026   0.0662   0.1324


In [7]:
!zip -r positioning_and_generality_results.zip results/full_positioning results/theil_diagnosis results/rate_barrier
from google.colab import files
files.download('positioning_and_generality_results.zip')

  adding: results/full_positioning/ (stored 0%)
  adding: results/full_positioning/fashion_mnist_seed0.json (deflated 72%)
  adding: results/full_positioning/fashion_mnist_seed1.json (deflated 72%)
  adding: results/full_positioning/fashion_mnist_seed2.json (deflated 72%)
  adding: results/full_positioning/fashion_mnist_summary.json (deflated 76%)
  adding: results/theil_diagnosis/ (stored 0%)
  adding: results/theil_diagnosis/fashion_mnist_seed0.json (deflated 69%)
  adding: results/rate_barrier/ (stored 0%)
  adding: results/rate_barrier/fashion_mnist_seed0.json (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>